# 07 · Skills – J.A.R.V.I.S. learns a procedure 🧪

> 🧪 **Preview.** Skills and toolboxes are preview features: names and behaviour may change. This notebook uses `AIProjectClient(..., allow_preview=True)`.

A **tool** is something an agent can *call*. A **skill** is something an agent *knows how to do*: a folder with a `SKILL.md` (YAML front matter + Markdown instructions, the open [Agent Skills](https://agentskills.io) format). Skills keep rules like "how we escalate incidents" **out of every agent's prompt** and in one versioned place.

| Delivery mode | How the agent gets the skill | Status in this repo |
|---|---|---|
| **Direct injection** | skill is downloaded into the agent project; the agent code adds it to its instructions | ✅ `jarvis-hosted` (chapter 09) |
| **Agent Framework `SkillsProvider`** | the model sees each skill's name and description and loads the body with `load_skill` when it fits | ❓ local sample [samples/skills_provider.py](../samples/skills_provider.py), run only against a scripted model |
| **Toolbox** | skill is attached to a toolbox; MCP clients read it as an MCP *resource* | ✅ shown below |
| **Prompt agent + GitHub Copilot harness** | skill reference in the agent definition | 🧪 code in section 7 (needs a subscription enrolled in the gated preview) |

Skill versions are immutable; the skill tracks a `default_version`. Deep dive: [docs/mcp-skills.md](../docs/mcp-skills.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

import re
from pathlib import Path
from azure.ai.projects.models import SkillInlineContent

skills = project(preview=True).beta.skills
SKILL_FILE = ROOT / "skills" / "stark-escalation" / "SKILL.md"
text = SKILL_FILE.read_text(encoding="utf-8")
front, body = re.match(r"---\n(.*?)\n---\n(.*)", text, re.S).groups()
meta = dict(line.split(": ", 1) for line in front.splitlines())
print(meta["name"], "-", meta["description"])

## 1 · Author and upload (version 1)
`name` must be lowercase letters, digits and hyphens. Creating a version creates the skill if it does not exist yet.

In [ ]:
v1 = skills.create(name=meta["name"], inline_content=SkillInlineContent(description=meta["description"], instructions=body.strip()))
print("created", v1.name, "version", v1.version)

## 2 · A new version, then promote it
Edit the policy (here: critical incidents must also name a floor), create **version 2**, test, then make it the default – agents and toolboxes that follow the default pick it up **without any code change**.

In [ ]:
body2 = body.strip() + "\n\n## Addendum\n6. For critical incidents, also state which floor of the tower is affected.\n"
v2 = skills.create(name=meta["name"], inline_content=SkillInlineContent(description=meta["description"], instructions=body2))
print("created version", v2.version)

print("before:", skills.get(name=meta["name"]).default_version)
skills.update(meta["name"], default_version=v2.version)
print("after :", skills.get(name=meta["name"]).default_version)

for v in skills.list_versions(meta["name"]):
    print("  version", v.version)

## 3 · Download (direct injection)
The hosted agent (`src/jarvis-hosted`) loads every `skills/<name>/SKILL.md` into its instructions at startup. `scripts/sync_skills.py` copies the repo's `skills/` there on deploy; you can also pull a skill **from Foundry**:

In [ ]:
import io, zipfile
data = b"".join(skills.download(meta["name"]))   # download_version(name, version) pins one version
with zipfile.ZipFile(io.BytesIO(data)) as z:
    print(z.namelist())
    target = ROOT / "src" / "jarvis-hosted" / "skills" / meta["name"]
    target.mkdir(parents=True, exist_ok=True)
    for n in z.namelist():
        if n.endswith("SKILL.md"):
            (target / "SKILL.md").write_bytes(z.read(n))
print("saved to", target)

## 4 · Attach to a toolbox
A **toolbox** is a named, versioned bundle of tools (and skills) behind **one MCP endpoint**. Attach the skill; omit `version` to follow the default.

In [ ]:
from azure.ai.projects.models import ToolboxSkillReference
tb = project(preview=True).toolboxes.create_version(
    name="stark-toolbox",
    description="Escalation know-how for the workshop",
    tools=[],
    skills=[ToolboxSkillReference(name=meta["name"])],
)
print("toolbox version:", tb.version)
TOOLBOX_URL = f"{cfg('STARK_PROJECT_ENDPOINT')}/toolboxes/stark-toolbox/versions/{tb.version}/mcp?api-version=v1"
print(TOOLBOX_URL)

## 5 · Read the skill like any MCP client
Skills appear as **MCP resources** (`resources/list`, `resources/read`). Any MCP client can read them – no Foundry SDK needed. (Toolbox versions must be published/promoted to be served on the consumer endpoint; the version-specific URL above is for testing.)

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

token = credential().get_token("https://ai.azure.com/.default").token
headers = {"Authorization": f"Bearer {token}", "Foundry-Features": "Skills=V1Preview"}

async with streamablehttp_client(TOOLBOX_URL, headers=headers) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        resources = await session.list_resources()
        for res in resources.resources:
            print("resource:", res.uri, "-", res.name)

## 6 · Is the skill really used? A marker test
Section 5 proves the skill can be **found**. It does not prove a model **follows** it. A toolbox only offers the skill: the client has to call `resources/read` and put the text in front of the model. Agent Framework (`SkillsProvider` with `MCPSkillsSource` 🧪) does this lazily: it advertises name and description, and loads the body only when the model decides the skill fits. So test the behaviour, not the attachment.

The skill leaves two markers in an answer:

| Marker | Comes from |
|---|---|
| answer starts with `Severity: <level>` | version 1, step 2 of the policy |
| a critical incident names a floor | the version 2 addendum, now the default |

Ask the same question twice: without the skill, and with the `SKILL.md` that the toolbox serves.

In [ ]:
async with streamablehttp_client(TOOLBOX_URL, headers=headers) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        found = [r.uri for r in (await session.list_resources()).resources if meta["name"] in str(r.uri)]
        uri = next((u for u in found if str(u).endswith("SKILL.md")), found[0])
        served = (await session.read_resource(uri)).contents[0].text
served_body = re.sub(r"^---\s*\n.*?\n---\s*\n", "", served, count=1, flags=re.S).strip()
print("read", uri, "-", len(served_body), "chars, has the version 2 addendum:", "Addendum" in served_body)

oai = openai_client()
QUESTION = "A suit just started smoking in the hangar. What do I do?"

def skill_markers(reply):
    return {"severity first": bool(re.match(r"\W*severity:", reply, re.I)), "names a floor": "floor" in reply.lower()}

for label, instructions in [("without the skill", "You are J.A.R.V.I.S."),
                            ("skill from the toolbox", "You are J.A.R.V.I.S.\n\n" + served_body)]:
    reply = oai.responses.create(model=model("chat"), instructions=instructions, input=QUESTION, store=False).output_text
    print("==", label, skill_markers(reply))
    print("  ", reply[:200].replace(chr(10), " "))

Both markers only with the skill: the text the toolbox serves reaches the model and changes its answer. One call is a sample, so run it a few times, or turn the marker into an evaluation criterion (chapter 10).

Use the same test on your own agent:

| Consumer of the toolbox | Is the skill used? | How to check |
|---|---|---|
| your code reads it and adds it to the instructions (this section) | ✅ always | marker test |
| Agent Framework `SkillsProvider` with `MCPSkillsSource` 🧪 | only when the model loads it | marker test, plus the `resources/read` call in the traces |
| prompt agent with an `MCPTool` on the toolbox URL | ❓ not documented, not tested here | marker test before you rely on it |

## 7 · Prompt agents + skills (GitHub Copilot harness, preview) 🧪
A prompt agent can list skills in its definition: `skills=[SkillReference(name=...)]`. But the skill only takes effect on the **GitHub Copilot harness**, an optional managed runtime that Foundry operates for prompt agents. The harness is chosen when the agent is created and cannot be changed later. Each agent version pins an immutable skill version.

| Mode | How the skill reaches the model |
|---|---|
| Direct injection (chapter 09) | your code adds `SKILL.md` to the instructions |
| Agent Framework `SkillsProvider` | the model loads it with `load_skill` when it fits ([samples/skills_provider.py](../samples/skills_provider.py)) |
| Toolbox | an MCP client reads it as a resource |
| **Copilot harness** | a skill reference in the agent definition |

The cell below creates the **same agent twice**, once on the default runtime and once on the harness, and asks both the same question. The skill says to answer with `Severity: <level>` first.

> ⚠️ The harness is a **gated preview**: Foundry answers `not enabled for this subscription` if your subscription is not enrolled. The cell then skips it and you only see the default-runtime result.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition, GitHubCopilotHarness, SkillReference
from azure.core.exceptions import HttpResponseError

section7_agents = ["jarvis-nb07-default", "jarvis-nb07-harness"]
section7_conversations = []

for name in section7_agents:
    options = dict(model=model("chat"), instructions="You are J.A.R.V.I.S. Use your skills when they match.",
                   skills=[SkillReference(name=meta["name"])])
    if name.endswith("harness"):
        options["harness"] = GitHubCopilotHarness()
    print("==", name)
    try:
        agent7 = project(preview=True).agents.create_version(agent_name=name, definition=PromptAgentDefinition(**options))
    except HttpResponseError as exc:
        print("   cannot create:", str(exc).splitlines()[0][:150])
        continue
    conv7 = oai.conversations.create()     # the harness needs a first-class conversation
    section7_conversations.append(conv7.id)
    reply = oai.responses.create(conversation=conv7.id, input=QUESTION, extra_body=agent_ref(agent7)).output_text
    print("   follows the skill:", "Severity:" in reply)
    print("  ", reply[:240].replace(chr(10), " "))

On the default runtime the skill reference is accepted but the answer shows no sign of the skill: it does not start with `Severity:`. Listing a skill is not enough, the runtime has to support it.

The same agent in an `azure.yaml` (from the Microsoft Learn docs; not run here):

```yaml
services:
  stark-skill:
    host: azure.ai.skill
    uses: [ai-project]
    archive: ./skills/stark-escalation
  my-agent:
    host: azure.ai.agent
    uses: [ai-project, stark-skill]
    kind: prompt
    harness:
      type: github_copilot_preview
    skills: [stark-skill]
```

> ℹ️ Concept: [Use the GitHub Copilot harness with prompt agents](https://learn.microsoft.com/azure/foundry/agents/how-to/use-tools-skills-copilot-harness). The harness also brings built-in file, shell, web-fetch and subagent tools, so review them before enabling broad capabilities.

## 👀 In the portal
🧪 Skills and the `stark-toolbox` toolbox are preview. The agent page shows the toolbox; where skills are listed was not checked.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
# keep the skill if you continue to chapter 09; uncomment to remove everything created here
# skills.delete(name=meta["name"])

for name in section7_agents:
    delete_agent(name)
for cid in section7_conversations:
    oai.conversations.delete(cid)

## 🎯 Recap
Skills move **procedures** out of prompts: author once, version, promote, deliver to many agents. Tools do things; skills teach how. To draft a new skill, run [samples/skills_provider.py](../samples/skills_provider.py) and check which skill the model loads.

> 💡 Next: [08 · Agents from YAML](08_agents_from_yaml.ipynb).